# Modelo 2 — Previsão Trimestral de Pagamentos por Órgão

Treina e avalia os regressores por quantil de `models/payment_forecast.py` (XGBoost, `reg:quantileerror`) sobre a Gold real, via Trino.

**Fonte de dado:** `iceberg.gold.fato_ordem_bancaria` — o pagamento efetivo ao credor (3º estágio da despesa: contrato → empenho → ordem bancária), criado em 24/07/2026 (ver `docs/06-analise-critica.md`, item 3). Antes desse fato existir na Gold, este modelo usava `fato_empenho` (compromisso orçamentário, não o desembolso) como proxy — a troca só mudou a query de origem, o resto do pipeline de features é o mesmo.

Pré-requisito: stack do lakehouse no ar (Hive Metastore + Trino, Gold já construída).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
load_dotenv(dotenv_path=project_root / ".env")

from models import payment_forecast as pf

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


## 1. Extração — série de ordens bancárias por órgão/trimestre + vigência de contratos


In [ ]:
pagamentos = pf.extract_pagamento_series()
contratos = pf.extract_contratos_vigencia()
print(f"Linhas de pagamento (OB): {len(pagamentos)} · órgãos distintos: {pagamentos['sk_orgao'].nunique()}")
print(f"Contratos com vigência: {len(contratos)}")
pagamentos.head()


## 2. Painel trimestral (órgão × trimestre)

`build_quarterly_panel` agrega o valor por trimestre, calcula `valor_contratado_ativo` (soma de contratos vigentes naquele trimestre), a flag de ano eleitoral e os lags (`lag_1_trimestre`, `lag_4_trimestres`). O alvo (`target_proximo_trimestre`) é o valor do **próximo** trimestre do mesmo órgão — é isso que o modelo aprende a prever.


In [ ]:
panel = pf.build_quarterly_panel(pagamentos, contratos)
print(f"Linhas do painel: {len(panel)}")
panel.sort_values(["sk_orgao", "ano", "trimestre"]).head(10)


In [ ]:
print("Trimestres cobertos:")
print(panel[["ano", "trimestre"]].drop_duplicates().sort_values(["ano", "trimestre"]))


## 3. Matriz de features


In [ ]:
X = pf.build_feature_matrix(panel)
y = panel.loc[X.index, "target_proximo_trimestre"]
print(f"Shape da matriz de features: {X.shape}")
print(f"Linhas com alvo conhecido (treináveis): {y.notna().sum()}")
X.describe().T


## 4. Avaliação — holdout temporal

O último trimestre com alvo conhecido vira teste; o resto treina — evita vazamento (nunca treina com dado "do futuro" em relação ao holdout). Métricas: MAE da mediana (p50) e cobertura do intervalo [p10, p90], que deveria rondar 80% se os quantis estiverem bem calibrados.


In [ ]:
metrics = pf.evaluate(panel, X)
metrics


## 5. Modelo final — treinado com todo o histórico conhecido


In [ ]:
train_idx = X.index[y.notna()]
models = pf.train_models(X.loc[train_idx], y.loc[train_idx])
preds_treino = pf.predict_quantiles(models, X.loc[train_idx])
print("Amplitude média do intervalo (p90 - p10):", (preds_treino["p90"] - preds_treino["p10"]).mean())


## 6. Previsão do próximo trimestre por órgão

Aplica o modelo às linhas cujo alvo ainda não existe (o trimestre mais recente de cada órgão) — a previsão real pedida pelo enunciado.


In [ ]:
resultado = pf.forecast_next_quarter(panel, X, models)
print(f"Órgãos com previsão: {len(resultado)}")
resultado.sort_values("valor_previsto_p50", ascending=False).head(20)


## 7. Persistência do modelo

Salva os 3 regressores (p10/p50/p90) em `models/artifacts/` — git-ignorado, reproduzível a partir deste notebook ou de `python -m models.payment_forecast`.


In [ ]:
pf.save_model(models, list(X.columns))
print(f"Modelo salvo em: {pf.ARTIFACT_PATH}")


## Achados e limitações

- **`tipo_despesa` via `natureza` orçamentária** — a feature `modalidade` no painel vem de `fato_ordem_bancaria.natureza` (classificação orçamentária padrão, ex: "3.3.90.18"), não de uma modalidade de contratação — mais fiel ao que o enunciado pede ("tipo de despesa") do que quando o proxy era `fato_empenho`.
- **Ordens canceladas excluídas** — `PAGAMENTO_QUERY` filtra `NOT flag_cancelada`, possível só depois que `fato_ordem_bancaria` passou a existir na Gold com esse campo.
- **Ano eleitoral simplificado** — todo ano par conta como eleitoral (municipal ou estadual/federal); não distingue o tipo de eleição.
- **Órgãos com pouco histórico** — o painel exige pelo menos um trimestre anterior (`lag_1_trimestre`) para entrar no treino; órgãos muito novos ou com poucos pagamentos ficam de fora da previsão até acumularem histórico.
- A produção contínua (re-treino + gravação em `iceberg.gold.previsao_pagamento_orgao`) roda pela DAG `dags/dag_ml_inference.py` — este notebook é só para treino/avaliação exploratórios, não escreve na Gold.
